In [1]:
import langchain
import chromadb
import pypdf
import os

DOCS_DIR = r"../documents"
CHROMA_DIR = r"../chroma_db"

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()

C:\Users\Dhyey\AppData\Local\Temp\ipykernel_34736\4268568213.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\Dhyey\anaconda3\envs\ml-ops\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

In [4]:
import hashlib
def log_chunk_id(doc):
    source = doc.metadata.get("source", "")
    page = doc.metadata.get("page", "")
    h = hashlib.sha256(f"{source}-{page}-{doc.page_content}".encode()).hexdigest()
    return h

ids = [log_chunk_id(c) for c in chunks]

In [5]:
from langchain_voyageai import VoyageAIEmbeddings
from langchain_chroma import Chroma

embedding_model = VoyageAIEmbeddings(model="voyage-4-lite")
vector_store = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model
)

In [6]:
existing = set(vector_store.get()["ids"])
new_chunks = [c for c, i in zip(chunks, ids) if i not in existing]
new_ids = [i for i in ids if i not in existing]

if new_chunks:
    vector_store.add_documents(documents=new_chunks, ids=new_ids)
    print(f"Added {len(new_chunks)} new chunks.")
else:
    print("No new chunks to add.")

No new chunks to add.


In [7]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following pieces of context to answer the question at the end.
If you don't know the answer, say that you don't know.
Context: {context}
Question: {question}
""")

In [10]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [15]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# result = chain.invoke("List all the monster hunter games, their release dates, supported platforms, and sales.")
result = chain.invoke("what's are the highest selling monster hunter game, how many sales does it have?")
print(result)

Based on the provided context, the highest-selling game in the series is ***Monster Hunter: World*** (referred to as *World*). 

As of July 5, 2022, it had **21 million sales**, making it Capcom's single best-selling video game of all time.
